In [ ]:
!pip -q install gradio networkx matplotlib pandas numpy

import gradio as gr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx


KNOWLEDGE_BASE = {
    "DNA": ["Gene", "Mutation", "Sequence", "Replication"],
    "Gene": ["DNA", "Protein", "Expression", "Mutation"],
    "Protein": ["Gene", "Amino Acid", "Structure", "Function"],
    "Mutation": ["DNA", "Variant", "Gene", "Disease"],
    "Variant": ["Mutation", "Gene", "Pathogenicity", "Phenotype"],
    "Pathway": ["Gene", "Protein", "Reaction", "Disease"],
    "CRISPR": ["DNA", "Cas9", "Guide RNA", "Gene Editing"],
    "Cas9": ["CRISPR", "Guide RNA", "DNA", "Gene Editing"],
    "Guide RNA": ["CRISPR", "Cas9", "Target DNA"],
    "BLAST": ["Sequence", "Alignment", "Similarity", "E-value"],
    "E-value": ["BLAST", "Significance", "Alignment"],
    "RNA": ["DNA", "Transcription", "Protein"],
    "Transcript": ["RNA", "Gene", "Expression"],
    "Expression": ["Gene", "RNA", "Protein"],
    "Disease": ["Gene", "Variant", "Pathway", "Phenotype"]
}

DEFINITIONS = {
    "DNA": "DNA is the molecule that stores hereditary biological information.",
    "Gene": "A gene is a functional region of DNA associated with a biological product or trait.",
    "Protein": "Proteins are biological molecules whose structures enable specific cellular functions.",
    "Mutation": "A mutation is a change in a DNA sequence.",
    "Variant": "A variant is an observed difference in a DNA sequence.",
    "Pathway": "A biological pathway represents connected molecular processes or interactions.",
    "CRISPR": "CRISPR is a programmable genome-editing system used to target specific DNA sequences.",
    "Cas9": "Cas9 is a nuclease used in CRISPR systems to cut targeted DNA.",
    "Guide RNA": "Guide RNA helps direct the CRISPR-Cas9 system toward a target DNA sequence.",
    "BLAST": "BLAST is a sequence comparison tool used to identify similar biological sequences.",
    "E-value": "The E-value in BLAST estimates how many matches of similar quality could occur by chance.",
    "RNA": "RNA is a nucleic acid involved in information transfer and regulation.",
    "Transcript": "A transcript is an RNA molecule produced from a DNA template.",
    "Expression": "Gene expression is the process through which information in a gene produces a functional product.",
    "Disease": "A disease is a condition associated with abnormal biological structure or function."
}

MASTERY = {
    "DNA": 96,
    "Gene": 91,
    "Protein": 83,
    "Mutation": 78,
    "Variant": 51,
    "Pathway": 66,
    "CRISPR": 88,
    "Cas9": 73,
    "Guide RNA": 42,
    "BLAST": 81,
    "E-value": 35,
    "RNA": 69
}

# ============================================================
# CONCEPT DETECTION
# ============================================================

def extract_concepts(text):
    found = []
    text = text.lower()

    for concept in KNOWLEDGE_BASE:
        if concept.lower() in text:
            found.append(concept)

    return list(dict.fromkeys(found))


# ============================================================
# KNOWLEDGE GRAPH
# ============================================================

def build_graph(selected=None, nodes=None):

    graph = nx.Graph()

    if nodes is None:
        nodes = set(KNOWLEDGE_BASE.keys())
    else:
        nodes = set(nodes)

    if selected and selected in KNOWLEDGE_BASE:
        nodes.add(selected)
        nodes.update(KNOWLEDGE_BASE[selected])

    for node in nodes:
        graph.add_node(node)

    for source in list(nodes):

        if source in KNOWLEDGE_BASE:

            for destination in KNOWLEDGE_BASE[source]:

                if destination in nodes:
                    graph.add_edge(source, destination)

    return graph


def create_graph(selected=None, nodes=None, title="Concept Relationship Network"):

    graph = build_graph(
        selected=selected,
        nodes=nodes
    )

    figure, axis = plt.subplots(
        figsize=(12, 7)
    )

    if len(graph.nodes) > 0:

        positions = nx.spring_layout(
            graph,
            seed=42,
            k=1.5
        )

        nx.draw_networkx_edges(
            graph,
            positions,
            ax=axis,
            alpha=0.35,
            width=1.5
        )

        nx.draw_networkx_nodes(
            graph,
            positions,
            ax=axis,
            node_size=[
                2200 if node == selected else 1300
                for node in graph.nodes
            ]
        )

        nx.draw_networkx_labels(
            graph,
            positions,
            ax=axis,
            font_size=9
        )

    axis.set_title(
        title,
        fontsize=16
    )

    axis.axis("off")

    figure.tight_layout()

    return figure


# ============================================================
# DASHBOARD
# ============================================================

def dashboard():

    average_score = int(
        np.mean(
            list(MASTERY.values())
        )
    )

    weakest = min(
        MASTERY,
        key=MASTERY.get
    )

    strongest = max(
        MASTERY,
        key=MASTERY.get
    )

    figure = create_graph()

    summary = f"""
# NeuroScape AI

## Knowledge Intelligence Dashboard

| Metric | Value |
|---|---:|
| Knowledge Score | {average_score}% |
| Total Concepts | {len(MASTERY)} |
| Weakest Concept | {weakest} |
| Strongest Concept | {strongest} |

NeuroScape AI visualizes relationships between biological concepts
and provides learning intelligence based on concept mastery.
"""

    return summary, figure


# ============================================================
# ASK NEUROSCAPE
# ============================================================

def ask_neuroscape(question):

    if not question.strip():

        return (
            "Please enter a question.",
            None,
            ""
        )

    concepts = extract_concepts(question)

    if not concepts:

        return (
            "I could not identify a tracked concept.\n\n"
            "Try asking about DNA, genes, proteins, variants, "
            "CRISPR, BLAST or pathways.",
            None,
            ""
        )

    primary = concepts[0]

    answer = (
        f"## {primary}\n\n"
        f"{DEFINITIONS[primary]}\n\n"
        f"### Connected Concepts\n\n"
        f"{', '.join(KNOWLEDGE_BASE[primary])}"
    )

    figure = create_graph(
        selected=primary,
        title=f"Knowledge Network: {primary}"
    )

    detected = "Detected Concepts:\n\n" + "\n".join(
        f"- {concept}"
        for concept in concepts
    )

    return answer, figure, detected


# ============================================================
# KNOWLEDGE DECAY
# ============================================================

def knowledge_decay(days):

    rows = []

    for concept, score in MASTERY.items():

        retention = (
            score *
            np.exp(-0.025 * days)
        )

        retention = max(
            5,
            retention
        )

        rows.append({
            "Concept": concept,
            "Current Score": round(score, 1),
            "Predicted Retention": round(retention, 1),
            "Knowledge Loss": round(score - retention, 1)
        })

    dataframe = pd.DataFrame(rows)

    dataframe = dataframe.sort_values(
        "Predicted Retention"
    )

    figure, axis = plt.subplots(
        figsize=(12, 6)
    )

    axis.bar(
        dataframe["Concept"],
        dataframe["Predicted Retention"]
    )

    axis.set_ylim(
        0,
        100
    )

    axis.set_ylabel(
        "Predicted Retention (%)"
    )

    axis.set_title(
        f"Knowledge Retention After {days} Days"
    )

    axis.tick_params(
        axis="x",
        rotation=45
    )

    figure.tight_layout()

    weakest = dataframe.iloc[0]

    recommendation = (
        f"Recommended revision concept: "
        f"{weakest['Concept']}\n\n"
        f"Predicted retention: "
        f"{weakest['Predicted Retention']}%"
    )

    return dataframe, figure, recommendation


# ============================================================
# MASTERY ANALYSIS
# ============================================================

def mastery_analysis():

    dataframe = pd.DataFrame({
        "Concept": list(MASTERY.keys()),
        "Mastery": list(MASTERY.values())
    })

    dataframe = dataframe.sort_values(
        "Mastery"
    )

    figure, axis = plt.subplots(
        figsize=(11, 6)
    )

    axis.barh(
        dataframe["Concept"],
        dataframe["Mastery"]
    )

    axis.set_xlim(
        0,
        100
    )

    axis.set_xlabel(
        "Mastery (%)"
    )

    axis.set_title(
        "Knowledge Strength by Concept"
    )

    figure.tight_layout()

    return dataframe, figure


# ============================================================
# MISCONCEPTION LAB
# ============================================================

def misconception_check(explanation):

    if not explanation.strip():

        return (
            "Please write your explanation first.",
            ""
        )

    text = explanation.lower()

    warnings = []

    if (
        "cas9" in text
        and "itself" in text
        and "target" in text
    ):

        warnings.append(
            "Potential misconception: target recognition "
            "involves guide RNA and the surrounding molecular "
            "context; Cas9 is the nuclease component."
        )

    if (
        "gene" in text
        and "always" in text
        and "protein" in text
    ):

        warnings.append(
            "Potential misconception: some genes produce "
            "functional RNAs rather than proteins."
        )

    concepts = extract_concepts(
        explanation
    )

    if warnings:

        result = "\n\n".join(
            f"Potential issue: {warning}"
            for warning in warnings
        )

    else:

        result = (
            "No obvious misconception pattern "
            "was detected by this prototype."
        )

    detected = (
        "Detected Concepts:\n\n"
        +
        (
            "\n".join(
                f"- {concept}"
                for concept in concepts
            )
            if concepts
            else
            "No tracked concepts detected."
        )
    )

    return result, detected


# ============================================================
# LEARNING REPLAY
# ============================================================

def learning_replay(day):

    milestones = [

        (
            1,
            [
                "DNA",
                "Gene"
            ]
        ),

        (
            7,
            [
                "DNA",
                "Gene",
                "Protein",
                "Mutation"
            ]
        ),

        (
            14,
            [
                "DNA",
                "Gene",
                "Protein",
                "Mutation",
                "Variant",
                "Pathway"
            ]
        ),

        (
            21,
            [
                "DNA",
                "Gene",
                "Protein",
                "Mutation",
                "Variant",
                "Pathway",
                "CRISPR"
            ]
        ),

        (
            30,
            list(MASTERY.keys())
        )
    ]

    replay_nodes = milestones[0][1]

    for milestone_day, nodes in milestones:

        if day >= milestone_day:
            replay_nodes = nodes

    graph = build_graph(
        nodes=replay_nodes
    )

    figure = create_graph(
        nodes=replay_nodes,
        title=f"Knowledge Evolution - Day {day}"
    )

    summary = f"""
### Learning Replay

**Replay Day:** {day}

**Concepts:** {len(replay_nodes)}

**Relationships:** {graph.number_of_edges()}
"""

    return summary, figure


# ============================================================
# GRADIO INTERFACE
# ============================================================

CSS = """
body {
    background: #070b13;
}

.gradio-container {
    max-width: 1400px !important;
    background: #070b13 !important;
}

h1, h2, h3 {
    color: white !important;
}

p, label, span {
    color: #d5dbea !important;
}

button {
    border-radius: 10px !important;
}
"""

with gr.Blocks(
    title="NeuroScape AI",
    css=CSS
) as app:

    gr.Markdown(
        """
# NeuroScape AI

### AI Knowledge Intelligence Platform

Visualize biological concepts, explore relationships,
analyze learning strength, simulate knowledge decay,
detect misconceptions and replay knowledge growth.
"""
    )

    # --------------------------------------------------------
    # DASHBOARD
    # --------------------------------------------------------

    with gr.Tab("Knowledge Map"):

        dashboard_text = gr.Markdown()

        dashboard_graph = gr.Plot()

        refresh_dashboard = gr.Button(
            "Load Knowledge Dashboard"
        )

        refresh_dashboard.click(
            dashboard,
            outputs=[
                dashboard_text,
                dashboard_graph
            ]
        )

        gr.Markdown(
            "### Concept Mastery"
        )

        mastery_table = gr.Dataframe()

        mastery_graph = gr.Plot()

        mastery_button = gr.Button(
            "Analyze Mastery"
        )

        mastery_button.click(
            mastery_analysis,
            outputs=[
                mastery_table,
                mastery_graph
            ]
        )

    # --------------------------------------------------------
    # ASK NEUROSCAPE
    # --------------------------------------------------------

    with gr.Tab("Ask NeuroScape"):

        question = gr.Textbox(
            label="Ask a question",
            placeholder=
            "Example: How is CRISPR connected to DNA?"
        )

        ask_button = gr.Button(
            "Analyze Question"
        )

        answer = gr.Markdown()

        detected = gr.Markdown()

        question_graph = gr.Plot()

        ask_button.click(
            ask_neuroscape,
            inputs=question,
            outputs=[
                answer,
                question_graph,
                detected
            ]
        )

    # --------------------------------------------------------
    # KNOWLEDGE DECAY
    # --------------------------------------------------------

    with gr.Tab("Knowledge Decay"):

        days = gr.Slider(
            minimum=1,
            maximum=90,
            value=14,
            step=1,
            label="Days Without Revision"
        )

        decay_button = gr.Button(
            "Simulate Knowledge Decay"
        )

        decay_table = gr.Dataframe()

        decay_graph = gr.Plot()

        decay_recommendation = gr.Markdown()

        decay_button.click(
            knowledge_decay,
            inputs=days,
            outputs=[
                decay_table,
                decay_graph,
                decay_recommendation
            ]
        )

    # --------------------------------------------------------
    # MISCONCEPTION LAB
    # --------------------------------------------------------

    with gr.Tab("Misconception Lab"):

        explanation = gr.Textbox(
            label="Write your explanation",
            lines=6,
            placeholder=
            "Example: Cas9 itself decides which DNA sequence to cut."
        )

        misconception_button = gr.Button(
            "Check Understanding"
        )

        misconception_result = gr.Markdown()

        misconception_concepts = gr.Markdown()

        misconception_button.click(
            misconception_check,
            inputs=explanation,
            outputs=[
                misconception_result,
                misconception_concepts
            ]
        )

    # --------------------------------------------------------
    # LEARNING REPLAY
    # --------------------------------------------------------

    with gr.Tab("Learning Replay"):

        replay_day = gr.Slider(
            minimum=1,
            maximum=30,
            value=30,
            step=1,
            label="Replay Day"
        )

        replay_button = gr.Button(
            "Replay Knowledge Growth"
        )

        replay_summary = gr.Markdown()

        replay_graph = gr.Plot()

        replay_button.click(
            learning_replay,
            inputs=replay_day,
            outputs=[
                replay_summary,
                replay_graph
            ]
        )

    gr.Markdown(
        """
---

### NeuroScape AI

Built with Python, Gradio, NetworkX, NumPy, Pandas and Matplotlib.
"""
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_1842/1190814333.py:563: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9cf48067eb152323fd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
